<a href="https://colab.research.google.com/github/Manjot-Coder/IP_LabAssignments/blob/main/Assignment-9/All%20questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Q1

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io

# Step 1: Upload grayscale image
uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=np.uint8)

# Step 2: Add edge padding
padded = np.pad(image, 1, mode="edge")

# Step 3: Create LBP image
height, width = image.shape
lbp_image = np.zeros((height, width), dtype=np.uint8)

# Clockwise neighbours starting from top-left
neighbors = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, 1),
    (1, 1), (1, 0), (1, -1),
    (0, -1)
]

for i in range(height):
    for j in range(width):

        center = padded[i+1, j+1]
        binary_value = 0

        for bit, (dx, dy) in enumerate(neighbors):

            neighbor = padded[i+1+dx, j+1+dy]

            if neighbor >= center:
                binary_value += (1 << (7-bit))

        lbp_image[i, j] = binary_value

# Step 4: Display original and LBP image
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Grayscale Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(lbp_image, cmap="gray", vmin=0, vmax=255)
plt.title("Local Binary Pattern")
plt.axis("off")

plt.tight_layout()
plt.show()

# Step 5: Plot LBP histogram
plt.figure(figsize=(8, 4))
plt.hist(lbp_image.flatten(), bins=256, range=(0, 256))
plt.title("LBP Histogram")
plt.xlabel("LBP Values")
plt.ylabel("Frequency")
plt.show()

# Step 6: Save output
Image.fromarray(lbp_image).save("lbp_output.jpg")

print("LBP image saved successfully!")


KeyboardInterrupt: 

In [ ]:
#Q2

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import io

# ---------------------------------------
# STEP 1: Upload grayscale image
# ---------------------------------------

uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=float)

# ---------------------------------------
# STEP 2: User inputs
# ---------------------------------------

size = int(input("Enter window size (3 or 5): "))

if size not in [3, 5]:
    raise ValueError("Window size must be 3 or 5.")

gaussian_noise_sigma = float(
    input("Enter Gaussian noise standard deviation (e.g. 20): ")
)

speckle_noise_sigma = float(
    input("Enter Speckle noise standard deviation (e.g. 0.2): ")
)

gaussian_filter_sigma = float(
    input("Enter Gaussian filter sigma (e.g. 1.0): ")
)

bilateral_sigma_color = float(
    input("Enter Bilateral intensity sigma (e.g. 30): ")
)

bilateral_sigma_space = float(
    input("Enter Bilateral spatial sigma (e.g. 2): ")
)

if (gaussian_noise_sigma < 0 or speckle_noise_sigma < 0 or
    gaussian_filter_sigma <= 0 or bilateral_sigma_color <= 0 or
    bilateral_sigma_space <= 0):
    raise ValueError("Invalid filter or noise parameters.")

rng = np.random.default_rng()


# ---------------------------------------
# STEP 3: Generate Gaussian and Speckle noise
# ---------------------------------------

# Gaussian noise: Additive noise
gaussian_noise = rng.normal(
    0, gaussian_noise_sigma, image.shape
)

noisy_gaussian = np.clip(
    image + gaussian_noise, 0, 255
)

# Speckle noise: Multiplicative noise
speckle_noise = rng.normal(
    0, speckle_noise_sigma, image.shape
)

speckle_corruption = image * speckle_noise

noisy_speckle = np.clip(
    image + speckle_corruption, 0, 255
)


# ---------------------------------------
# STEP 4: Wiener Filter
# ---------------------------------------

def wiener_filter(img, size, noise_variance):

    pad = size // 2
    padded = np.pad(img, pad, mode="edge")

    result = np.zeros(img.shape)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):

            region = padded[i:i+size, j:j+size]

            local_mean = np.mean(region)
            local_variance = np.var(region)

            if local_variance > 0:
                factor = max(
                    local_variance - noise_variance, 0
                ) / local_variance
            else:
                factor = 0

            result[i, j] = (
                local_mean +
                factor * (img[i, j] - local_mean)
            )

    return np.clip(result, 0, 255)


# ---------------------------------------
# STEP 5: Median Filter
# ---------------------------------------

def median_filter(img, size):

    pad = size // 2
    padded = np.pad(img, pad, mode="edge")

    result = np.zeros(img.shape)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):

            region = padded[i:i+size, j:j+size]

            result[i, j] = np.median(region)

    return result


# ---------------------------------------
# STEP 6: Gaussian Filter
# ---------------------------------------

def gaussian_filter(img, size, sigma):

    pad = size // 2

    # Create Gaussian kernel
    kernel = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            x = i - pad
            y = j - pad

            kernel[i, j] = np.exp(
                -(x*x + y*y) / (2*sigma*sigma)
            )

    kernel = kernel / np.sum(kernel)

    padded = np.pad(img, pad, mode="edge")

    result = np.zeros(img.shape)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):

            region = padded[i:i+size, j:j+size]

            result[i, j] = np.sum(region * kernel)

    return np.clip(result, 0, 255)


# ---------------------------------------
# STEP 7: Bilateral Filter
# ---------------------------------------

def bilateral_filter(img, size, sigma_color, sigma_space):

    pad = size // 2
    padded = np.pad(img, pad, mode="edge")

    result = np.zeros(img.shape)

    # Spatial Gaussian weights
    spatial_weights = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            x = i - pad
            y = j - pad

            spatial_weights[i, j] = np.exp(
                -(x*x + y*y) / (2*sigma_space*sigma_space)
            )

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):

            region = padded[i:i+size, j:j+size]

            center = img[i, j]

            # Intensity difference
            intensity_difference = region - center

            range_weights = np.exp(
                -(intensity_difference**2) /
                (2*sigma_color*sigma_color)
            )

            # Combine spatial and intensity weights
            weights = spatial_weights * range_weights

            result[i, j] = np.sum(
                weights * region
            ) / np.sum(weights)

    return np.clip(result, 0, 255)


# ---------------------------------------
# STEP 8: Apply filters on Gaussian noisy image
# ---------------------------------------

# Known noise variance because noise was generated here
gaussian_variance = np.var(gaussian_noise)

wiener_gaussian = wiener_filter(
    noisy_gaussian, size, gaussian_variance
)

median_gaussian = median_filter(
    noisy_gaussian, size
)

gaussian_filtered = gaussian_filter(
    noisy_gaussian, size, gaussian_filter_sigma
)

bilateral_gaussian = bilateral_filter(
    noisy_gaussian, size,
    bilateral_sigma_color, bilateral_sigma_space
)


# ---------------------------------------
# STEP 9: Apply filters on Speckle noisy image
# ---------------------------------------

speckle_variance = np.var(speckle_corruption)

wiener_speckle = wiener_filter(
    noisy_speckle, size, speckle_variance
)

median_speckle = median_filter(
    noisy_speckle, size
)

gaussian_speckle = gaussian_filter(
    noisy_speckle, size, gaussian_filter_sigma
)

bilateral_speckle = bilateral_filter(
    noisy_speckle, size,
    bilateral_sigma_color, bilateral_sigma_space
)


# ---------------------------------------
# STEP 10: Display Gaussian noise results
# ---------------------------------------

gaussian_results = [
    noisy_gaussian,
    wiener_gaussian,
    median_gaussian,
    gaussian_filtered,
    bilateral_gaussian
]

titles = [
    "Noisy Image",
    "Wiener Filter",
    "Median Filter",
    "Gaussian Filter",
    "Bilateral Filter"
]

plt.figure(figsize=(17, 5))

for i in range(5):

    plt.subplot(1, 5, i+1)
    plt.imshow(gaussian_results[i], cmap="gray", vmin=0, vmax=255)
    plt.title(titles[i])
    plt.axis("off")

plt.suptitle("Denoising Gaussian Noise")
plt.tight_layout()
plt.show()


# ---------------------------------------
# STEP 11: Display Speckle noise results
# ---------------------------------------

speckle_results = [
    noisy_speckle,
    wiener_speckle,
    median_speckle,
    gaussian_speckle,
    bilateral_speckle
]

plt.figure(figsize=(17, 5))

for i in range(5):

    plt.subplot(1, 5, i+1)
    plt.imshow(speckle_results[i], cmap="gray", vmin=0, vmax=255)
    plt.title(titles[i])
    plt.axis("off")

plt.suptitle("Denoising Speckle Noise")
plt.tight_layout()
plt.show()


# ---------------------------------------
# STEP 12: Save all output images
# ---------------------------------------

results = {
    "wiener_gaussian.jpg": wiener_gaussian,
    "median_gaussian.jpg": median_gaussian,
    "gaussian_filtered.jpg": gaussian_filtered,
    "bilateral_gaussian.jpg": bilateral_gaussian,

    "wiener_speckle.jpg": wiener_speckle,
    "median_speckle.jpg": median_speckle,
    "gaussian_speckle.jpg": gaussian_speckle,
    "bilateral_speckle.jpg": bilateral_speckle
}

for name, result in results.items():

    Image.fromarray(
        result.astype(np.uint8)
    ).save(name)

print("All denoised images saved successfully!")
